# Stopwords, Vocabulary & Phrase Matching
### NLP Practical using *Pinocchio*

**Libraries:** spaCy, NLTK  
**Text source:** `pinocchio.txt`

This notebook combines stopword processing and rule-based matching. All examples use the supplied story extract.

## 1. Setup
Install the libraries and English language model once if needed.

In [1]:
%pip install spacy nltk
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 25.2 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import spacy
import nltk
from nltk.corpus import stopwords
from spacy.matcher import Matcher, PhraseMatcher

nlp = spacy.load("en_core_web_sm")
nltk.download("stopwords", quiet=True)

# Read the story used throughout this notebook.
with open("pinocchio.txt", "r", encoding="utf-8") as file:
    story = file.read()

doc = nlp(story)
print(story)

The yelling was of no use, for the Marionettes, instead of
going on with their act, made twice as much racket as
before, and, lifting up Pinocchio on their shoulders,
carried him around the stage in triumph.
At that very moment, the Director came out of his room. He
had such a fearful appearance that one look at him would
fill you with horror. His beard was as black as pitch, and
so long that it reached from his chin down to his feet. His
mouth was as wide as an oven, his teeth like yellow fangs,
and his eyes, two glowing red coals. In his huge, hairy
hands, a long whip, made of green snakes and black cats'
tails twisted together, swished through the air in a
dangerous way.
At the unexpected apparition, no one dared even to breathe.
One could almost hear a fly go by. Those poor Marionettes,
one and all, trembled like leaves in a storm.
"Why have you brought such excitement into my theater;" the
huge fellow asked Pinocchio with the voice of an ogre
suffering with a cold.
"Believe me, yo

## 2. Stopwords using spaCy
Stopwords are common words that may be removed for some NLP tasks. The following operations work on the story text.

In [3]:
# Check whether the words in the story are stopwords.
for token in doc:
    if token.is_alpha:
        print(token.text, "->", token.is_stop)

The -> True
yelling -> False
was -> True
of -> True
no -> True
use -> False
for -> True
the -> True
Marionettes -> False
instead -> False
of -> True
going -> False
on -> True
with -> True
their -> True
act -> False
made -> True
twice -> False
as -> True
much -> True
racket -> False
as -> True
before -> True
and -> True
lifting -> False
up -> True
Pinocchio -> False
on -> True
their -> True
shoulders -> False
carried -> False
him -> True
around -> True
the -> True
stage -> False
in -> True
triumph -> False
At -> True
that -> True
very -> True
moment -> False
the -> True
Director -> False
came -> False
out -> True
of -> True
his -> True
room -> False
He -> True
had -> True
such -> True
a -> True
fearful -> False
appearance -> False
that -> True
one -> True
look -> False
at -> True
him -> True
would -> True
fill -> False
you -> True
with -> True
horror -> False
His -> True
beard -> False
was -> True
as -> True
black -> False
as -> True
pitch -> False
and -> True
so -> True
long -> False
t

In [4]:
# Extract stopwords from the story.
spa_stopwords = [token.text for token in doc if token.is_stop and token.is_alpha]
print(spa_stopwords)

['The', 'was', 'of', 'no', 'for', 'the', 'of', 'on', 'with', 'their', 'made', 'as', 'much', 'as', 'before', 'and', 'up', 'on', 'their', 'him', 'around', 'the', 'in', 'At', 'that', 'very', 'the', 'out', 'of', 'his', 'He', 'had', 'such', 'a', 'that', 'one', 'at', 'him', 'would', 'you', 'with', 'His', 'was', 'as', 'as', 'and', 'so', 'that', 'it', 'from', 'his', 'down', 'to', 'his', 'His', 'was', 'as', 'as', 'an', 'his', 'and', 'his', 'two', 'In', 'his', 'a', 'made', 'of', 'and', 'together', 'through', 'the', 'in', 'a', 'At', 'the', 'no', 'one', 'even', 'to', 'One', 'could', 'almost', 'a', 'go', 'by', 'Those', 'one', 'and', 'all', 'in', 'a', 'Why', 'have', 'you', 'such', 'into', 'my', 'the', 'with', 'the', 'of', 'an', 'with', 'a', 'me', 'your', 'the', 'was', 'not', 'mine', 'Enough', 'Be', 'I', 'take', 'of', 'you', 'As', 'as', 'the', 'was', 'over', 'the', 'to', 'the', 'where', 'a', 'was', 'on', 'the', 'More', 'was', 'to', 'it', 'He', 'and', 'and', 'to', 'them', 'that', 'to', 'me', 'He', 'as

In [5]:
# Extract non-stopwords from the story.
non_stopwords = [token.text for token in doc if not token.is_stop and token.is_alpha]
print(non_stopwords)

['yelling', 'use', 'Marionettes', 'instead', 'going', 'act', 'twice', 'racket', 'lifting', 'Pinocchio', 'shoulders', 'carried', 'stage', 'triumph', 'moment', 'Director', 'came', 'room', 'fearful', 'appearance', 'look', 'fill', 'horror', 'beard', 'black', 'pitch', 'long', 'reached', 'chin', 'feet', 'mouth', 'wide', 'oven', 'teeth', 'like', 'yellow', 'fangs', 'eyes', 'glowing', 'red', 'coals', 'huge', 'hairy', 'hands', 'long', 'whip', 'green', 'snakes', 'black', 'cats', 'tails', 'twisted', 'swished', 'air', 'dangerous', 'way', 'unexpected', 'apparition', 'dared', 'breathe', 'hear', 'fly', 'poor', 'Marionettes', 'trembled', 'like', 'leaves', 'storm', 'brought', 'excitement', 'theater', 'huge', 'fellow', 'asked', 'Pinocchio', 'voice', 'ogre', 'suffering', 'cold', 'Believe', 'Honor', 'fault', 'quiet', 'care', 'later', 'soon', 'play', 'Director', 'went', 'kitchen', 'fine', 'big', 'lamb', 'slowly', 'turning', 'spit', 'wood', 'needed', 'finish', 'cooking', 'called', 'Harlequin', 'Pulcinella', 

In [6]:
# Remove stopwords while keeping punctuation.
result = [token.text for token in doc if not token.is_stop]
print(" ".join(result))

yelling use , Marionettes , instead 
 going act , twice racket 
 , , lifting Pinocchio shoulders , 
 carried stage triumph . 
 moment , Director came room . 
 fearful appearance look 
 fill horror . beard black pitch , 
 long reached chin feet . 
 mouth wide oven , teeth like yellow fangs , 
 eyes , glowing red coals . huge , hairy 
 hands , long whip , green snakes black cats ' 
 tails twisted , swished air 
 dangerous way . 
 unexpected apparition , dared breathe . 
 hear fly . poor Marionettes , 
 , trembled like leaves storm . 
 " brought excitement theater ; " 
 huge fellow asked Pinocchio voice ogre 
 suffering cold . 
 " Believe , Honor , fault . " 
 " ! quiet ! care later . " 
 soon play , Director went 
 kitchen , fine big lamb slowly turning 
 spit . wood needed finish cooking . called 
 Harlequin Pulcinella said : 
 " Bring Marionette ! looks 
 - seasoned wood . fine fire 
 spit . " 
 Harlequin Pulcinella hesitated bit . , frightened 
 look master , left kitchen obey 
 . min

In [7]:
# Remove stopwords and punctuation.
result = [token.text for token in doc if not token.is_stop and not token.is_punct]
print(" ".join(result))

yelling use Marionettes instead 
 going act twice racket 
 lifting Pinocchio shoulders 
 carried stage triumph 
 moment Director came room 
 fearful appearance look 
 fill horror beard black pitch 
 long reached chin feet 
 mouth wide oven teeth like yellow fangs 
 eyes glowing red coals huge hairy 
 hands long whip green snakes black cats 
 tails twisted swished air 
 dangerous way 
 unexpected apparition dared breathe 
 hear fly poor Marionettes 
 trembled like leaves storm 
 brought excitement theater 
 huge fellow asked Pinocchio voice ogre 
 suffering cold 
 Believe Honor fault 
 quiet care later 
 soon play Director went 
 kitchen fine big lamb slowly turning 
 spit wood needed finish cooking called 
 Harlequin Pulcinella said 
 Bring Marionette looks 
 seasoned wood fine fire 
 spit 
 Harlequin Pulcinella hesitated bit frightened 
 look master left kitchen obey 
 minutes later returned carrying poor 
 Pinocchio wriggling squirming like eel 
 crying pitifully 
 Father save want d

In [8]:
# Convert the story to lowercase and remove stopwords.
result = [token.text.lower() for token in doc if not token.is_stop and token.is_alpha]
print(" ".join(result))

yelling use marionettes instead going act twice racket lifting pinocchio shoulders carried stage triumph moment director came room fearful appearance look fill horror beard black pitch long reached chin feet mouth wide oven teeth like yellow fangs eyes glowing red coals huge hairy hands long whip green snakes black cats tails twisted swished air dangerous way unexpected apparition dared breathe hear fly poor marionettes trembled like leaves storm brought excitement theater huge fellow asked pinocchio voice ogre suffering cold believe honor fault quiet care later soon play director went kitchen fine big lamb slowly turning spit wood needed finish cooking called harlequin pulcinella said bring marionette looks seasoned wood fine fire spit harlequin pulcinella hesitated bit frightened look master left kitchen obey minutes later returned carrying poor pinocchio wriggling squirming like eel crying pitifully father save want die want die


In [9]:
# Display the spaCy English stopword list and its size.
print(sorted(nlp.Defaults.stop_words))
print("Total stopwords:", len(nlp.Defaults.stop_words))

["'d", "'ll", "'m", "'re", "'s", "'ve", 'a', 'about', 'above', 'across', 'after', 'afterwards', 'again', 'against', 'all', 'almost', 'alone', 'along', 'already', 'also', 'although', 'always', 'am', 'among', 'amongst', 'amount', 'an', 'and', 'another', 'any', 'anyhow', 'anyone', 'anything', 'anyway', 'anywhere', 'are', 'around', 'as', 'at', 'back', 'be', 'became', 'because', 'become', 'becomes', 'becoming', 'been', 'before', 'beforehand', 'behind', 'being', 'below', 'beside', 'besides', 'between', 'beyond', 'both', 'bottom', 'but', 'by', 'ca', 'call', 'can', 'cannot', 'could', 'did', 'do', 'does', 'doing', 'done', 'down', 'due', 'during', 'each', 'eight', 'either', 'eleven', 'else', 'elsewhere', 'empty', 'enough', 'even', 'ever', 'every', 'everyone', 'everything', 'everywhere', 'except', 'few', 'fifteen', 'fifty', 'first', 'five', 'for', 'former', 'formerly', 'forty', 'four', 'from', 'front', 'full', 'further', 'get', 'give', 'go', 'had', 'has', 'have', 'he', 'hence', 'her', 'here', 'he

In [11]:
# Add a custom stopword taken from the story: Pinocchio.
# Use a separate Doc and token flag so later examples remain unchanged.
# Add "Pinocchio" as a custom stopword

nlp.vocab["pinocchio"].is_stop = True

# Process the story
custom_doc = nlp(story)

# Display all stopwords
print([token.text for token in custom_doc if token.is_stop and token.is_alpha])

['The', 'was', 'of', 'no', 'for', 'the', 'of', 'on', 'with', 'their', 'made', 'as', 'much', 'as', 'before', 'and', 'up', 'on', 'their', 'him', 'around', 'the', 'in', 'At', 'that', 'very', 'the', 'out', 'of', 'his', 'He', 'had', 'such', 'a', 'that', 'one', 'at', 'him', 'would', 'you', 'with', 'His', 'was', 'as', 'as', 'and', 'so', 'that', 'it', 'from', 'his', 'down', 'to', 'his', 'His', 'was', 'as', 'as', 'an', 'his', 'and', 'his', 'two', 'In', 'his', 'a', 'made', 'of', 'and', 'together', 'through', 'the', 'in', 'a', 'At', 'the', 'no', 'one', 'even', 'to', 'One', 'could', 'almost', 'a', 'go', 'by', 'Those', 'one', 'and', 'all', 'in', 'a', 'Why', 'have', 'you', 'such', 'into', 'my', 'the', 'with', 'the', 'of', 'an', 'with', 'a', 'me', 'your', 'the', 'was', 'not', 'mine', 'Enough', 'Be', 'I', 'take', 'of', 'you', 'As', 'as', 'the', 'was', 'over', 'the', 'to', 'the', 'where', 'a', 'was', 'on', 'the', 'More', 'was', 'to', 'it', 'He', 'and', 'and', 'to', 'them', 'that', 'to', 'me', 'He', 'as

In [12]:
# Function to remove stopwords from the story.
def remove_stopwords(text):
    doc = nlp(text)
    return " ".join(token.text for token in doc if not token.is_stop and token.is_alpha)

print(remove_stopwords(story))

yelling use Marionettes instead going act twice racket lifting Pinocchio shoulders carried stage triumph moment Director came room fearful appearance look fill horror beard black pitch long reached chin feet mouth wide oven teeth like yellow fangs eyes glowing red coals huge hairy hands long whip green snakes black cats tails twisted swished air dangerous way unexpected apparition dared breathe hear fly poor Marionettes trembled like leaves storm brought excitement theater huge fellow asked Pinocchio voice ogre suffering cold Believe Honor fault quiet care later soon play Director went kitchen fine big lamb slowly turning spit wood needed finish cooking called Harlequin Pulcinella said Bring Marionette looks seasoned wood fine fire spit Harlequin Pulcinella hesitated bit frightened look master left kitchen obey minutes later returned carrying poor Pinocchio wriggling squirming like eel crying pitifully Father save want die want die


## 3. Stopwords using NLTK
NLTK provides its own English stopword list. spaCy tokenization is reused to keep the code simple.

In [13]:
# Load and display NLTK English stopwords.
nltk_stopwords = set(stopwords.words("english"))
print(sorted(nltk_stopwords))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", "he's", 'her', 'here', 'hers', 'herself', 'him', 'himself', 'his', 'how', 'i', "i'd", "i'll", "i'm", "i've", 'if', 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [14]:
# Check which story words appear in the NLTK stopword list.
for token in doc:
    if token.is_alpha:
        print(token.text, "->", token.text.lower() in nltk_stopwords)

The -> True
yelling -> False
was -> True
of -> True
no -> True
use -> False
for -> True
the -> True
Marionettes -> False
instead -> False
of -> True
going -> False
on -> True
with -> True
their -> True
act -> False
made -> False
twice -> False
as -> True
much -> False
racket -> False
as -> True
before -> True
and -> True
lifting -> False
up -> True
Pinocchio -> False
on -> True
their -> True
shoulders -> False
carried -> False
him -> True
around -> False
the -> True
stage -> False
in -> True
triumph -> False
At -> True
that -> True
very -> True
moment -> False
the -> True
Director -> False
came -> False
out -> True
of -> True
his -> True
room -> False
He -> True
had -> True
such -> True
a -> True
fearful -> False
appearance -> False
that -> True
one -> False
look -> False
at -> True
him -> True
would -> False
fill -> False
you -> True
with -> True
horror -> False
His -> True
beard -> False
was -> True
as -> True
black -> False
as -> True
pitch -> False
and -> True
so -> True
long -> Fa

In [15]:
# Extract NLTK stopwords from the story.
result = [token.text for token in doc if token.is_alpha and token.text.lower() in nltk_stopwords]
print(result)

['The', 'was', 'of', 'no', 'for', 'the', 'of', 'on', 'with', 'their', 'as', 'as', 'before', 'and', 'up', 'on', 'their', 'him', 'the', 'in', 'At', 'that', 'very', 'the', 'out', 'of', 'his', 'He', 'had', 'such', 'a', 'that', 'at', 'him', 'you', 'with', 'His', 'was', 'as', 'as', 'and', 'so', 'that', 'it', 'from', 'his', 'down', 'to', 'his', 'His', 'was', 'as', 'as', 'an', 'his', 'and', 'his', 'In', 'his', 'a', 'of', 'and', 'through', 'the', 'in', 'a', 'At', 'the', 'no', 'to', 'a', 'by', 'Those', 'and', 'all', 'in', 'a', 'Why', 'have', 'you', 'such', 'into', 'my', 'the', 'with', 'the', 'of', 'an', 'with', 'a', 'me', 'your', 'the', 'was', 'not', 'Be', 'I', 'of', 'you', 'As', 'as', 'the', 'was', 'over', 'the', 'to', 'the', 'where', 'a', 'was', 'on', 'the', 'More', 'was', 'to', 'it', 'He', 'and', 'and', 'to', 'them', 'that', 'to', 'me', 'He', 'as', 'if', 'he', 'were', 'of', 'He', 'a', 'for', 'this', 'and', 'a', 'Then', 'by', 'a', 'from', 'their', 'they', 'the', 'to', 'him', 'A', 'few', 'they'

In [16]:
# Remove NLTK stopwords from the story.
result = [token.text for token in doc if token.is_alpha and token.text.lower() not in nltk_stopwords]
print(" ".join(result))

yelling use Marionettes instead going act made twice much racket lifting Pinocchio shoulders carried around stage triumph moment Director came room fearful appearance one look would fill horror beard black pitch long reached chin feet mouth wide oven teeth like yellow fangs eyes two glowing red coals huge hairy hands long whip made green snakes black cats tails twisted together swished air dangerous way unexpected apparition one dared even breathe One could almost hear fly go poor Marionettes one trembled like leaves storm brought excitement theater huge fellow asked Pinocchio voice ogre suffering cold Believe Honor fault mine Enough quiet take care later soon play Director went kitchen fine big lamb slowly turning spit wood needed finish cooking called Harlequin Pulcinella said Bring Marionette looks made well seasoned wood make fine fire spit Harlequin Pulcinella hesitated bit frightened look master left kitchen obey minutes later returned carrying poor Pinocchio wriggling squirming 

In [17]:
# Convert to lowercase and remove NLTK stopwords and punctuation.
result = [token.text.lower() for token in doc if token.is_alpha and token.text.lower() not in nltk_stopwords]
print(" ".join(result))

yelling use marionettes instead going act made twice much racket lifting pinocchio shoulders carried around stage triumph moment director came room fearful appearance one look would fill horror beard black pitch long reached chin feet mouth wide oven teeth like yellow fangs eyes two glowing red coals huge hairy hands long whip made green snakes black cats tails twisted together swished air dangerous way unexpected apparition one dared even breathe one could almost hear fly go poor marionettes one trembled like leaves storm brought excitement theater huge fellow asked pinocchio voice ogre suffering cold believe honor fault mine enough quiet take care later soon play director went kitchen fine big lamb slowly turning spit wood needed finish cooking called harlequin pulcinella said bring marionette looks made well seasoned wood make fine fire spit harlequin pulcinella hesitated bit frightened look master left kitchen obey minutes later returned carrying poor pinocchio wriggling squirming 

In [18]:
# Count NLTK stopwords in the story.
count = sum(1 for token in doc if token.is_alpha and token.text.lower() in nltk_stopwords)
print("Stopword count:", count)

Stopword count: 160


In [19]:
# Display the languages supported by NLTK stopwords.
print(stopwords.fileids())

['albanian', 'arabic', 'azerbaijani', 'basque', 'belarusian', 'bengali', 'catalan', 'chinese', 'danish', 'dutch', 'english', 'finnish', 'french', 'german', 'greek', 'hebrew', 'hinglish', 'hungarian', 'indonesian', 'italian', 'kazakh', 'nepali', 'norwegian', 'portuguese', 'romanian', 'russian', 'slovene', 'spanish', 'swedish', 'tajik', 'tamil', 'turkish', 'uzbek']


## 4. Vocabulary and rule-based matching
spaCy’s `Matcher` searches for token patterns in the story. Patterns below are taken directly from the supplied passage.

In [20]:
# Create a matcher using spaCy vocabulary.
matcher = Matcher(nlp.vocab)

# Match the character name and a phrase in the passage.
pattern1 = [{"LOWER": "pinocchio"}]
pattern2 = [{"LOWER": "poor"}, {"LOWER": "marionettes"}]
pattern3 = [{"LOWER": "black"}, {"LOWER": "cats"}]

matcher.add("StoryPatterns", [pattern1, pattern2, pattern3])

In [21]:
# Find matching patterns in the story.
found_matches = matcher(doc)
print(found_matches)

[(904650475551871745, 33, 34), (904650475551871745, 146, 148), (904650475551871745, 188, 190), (904650475551871745, 221, 222), (904650475551871745, 387, 388)]


In [22]:
# Display each match with its label and token positions.
for match_id, start, end in found_matches:
    label = nlp.vocab.strings[match_id]
    print(label, start, end, doc[start:end].text)

StoryPatterns 33 34 Pinocchio
StoryPatterns 146 148 black cats
StoryPatterns 188 190 poor Marionettes
StoryPatterns 221 222 Pinocchio
StoryPatterns 387 388 Pinocchio


## 5. Phrase matching
`PhraseMatcher` locates exact phrases in the story. The phrases below all occur in `pinocchio.txt`.

In [23]:
# Create phrase patterns using expressions from the story.
phrase_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
phrases = ["Pinocchio", "Harlequin and Pulcinella", "the Director", "green snakes"]
patterns = [nlp.make_doc(phrase) for phrase in phrases]
phrase_matcher.add("PinocchioPhrases", patterns)

In [24]:
# Display phrases found in the story.
for match_id, start, end in phrase_matcher(doc):
    print(doc[start:end].text)

Pinocchio
the Director
green snakes
Pinocchio
the Director
Harlequin and Pulcinella
Harlequin and Pulcinella
Pinocchio


## 6. Conclusion
The practical demonstrated stopword identification and removal using spaCy and NLTK, followed by token-pattern matching and phrase matching using the same *Pinocchio* passage. Stopword removal is task-dependent: words important for meaning should not always be removed.

----